In [1]:
# =============================================================================
# STEP 6c - CIC-IDS2017 ADDED TO THE CLASS-CONDITIONAL SHIFT ANALYSIS
#
# Notebook 42 excluded CIC-IDS2017 because its source/target split is
# realisation-specific. That reason was weak: a per-realisation split only means
# computing per realisation and averaging, which is exactly how the existing CIC
# shift measures are already built, and the index files are on disk.
#
# Run AFTER notebook 43. This appends CIC to the class-conditional table and
# redoes the within-dataset comparison across all four environments.
#
# IMPORT ORDER MATTERS HERE. Third-party libraries are imported BEFORE any Drive
# path enters sys.path. If the Drive FUSE mount drops, Python's import machinery
# walks the dead path and fails on whatever it imports next, which produces an
# Errno 107 traceback pointing at sklearn rather than at Drive. Importing first
# and asserting the mount second turns that into a one-line diagnosis.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os, sys, json, shutil, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), (
    'Drive mount is not healthy. Restart the runtime (Runtime > Restart session), '
    'remount, and re-run. A remount alone can leave a stale path in sys.path.')

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','features_cic']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
import features_cic as fc
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
SUB=20000; FOLDS=5; MIN_PER_SIDE=40
print('ready | alpha', ALPHA, '| per-side cap', SUB, '| min rows per side', MIN_PER_SIDE)


Mounted at /content/drive
ready | alpha 0.05 | per-side cap 20000 | min rows per side 40


In [2]:
# =============================================================================
# Cell 2 - the same estimator and permutation null used in notebook 42, so the
# CIC rows are measured on the identical scale as the other three environments.
# =============================================================================
def scov(Xs, Xt, seed=0, folds=FOLDS):
    """Cross-fitted domain-classifier AUROC between two feature matrices."""
    ns_, nt_ = len(Xs), len(Xt)
    if ns_ < MIN_PER_SIDE or nt_ < MIN_PER_SIDE: return np.nan, ns_, nt_
    m = min(ns_, nt_, SUB); rg = np.random.default_rng(seed)
    X = np.vstack([Xs[rg.choice(ns_, m, replace=False)], Xt[rg.choice(nt_, m, replace=False)]])
    y = np.r_[np.zeros(m), np.ones(m)]
    k = min(folds, m // 10) if m < 10*folds else folds
    if k < 2: return np.nan, ns_, nt_
    aucs=[]
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(X, y):
        mod = HistGradientBoostingClassifier(max_iter=100, random_state=seed).fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], mod.predict_proba(X[te])[:, 1]))
    return float(np.mean(aucs)), ns_, nt_

def scov_null(Xs, Xt, seed=0, draws=5):
    """Permutation reference: split the POOLED per-class data at random, so this
    measures estimator optimism rather than genuine source/target separation."""
    if len(Xs) < MIN_PER_SIDE or len(Xt) < MIN_PER_SIDE: return np.nan
    P = np.vstack([Xs, Xt]); out=[]
    for d in range(draws):
        rg = np.random.default_rng(seed + 1000 + d)
        perm = rg.permutation(len(P)); half = len(P)//2
        v,_,_ = scov(P[perm[:half]], P[perm[half:]], seed=seed+d, folds=3)
        if not np.isnan(v): out.append(v)
    return float(np.mean(out)) if out else np.nan
print('estimator and permutation null defined (identical to notebook 42)')


estimator and permutation null defined (identical to notebook 42)


In [3]:
# =============================================================================
# Cell 3 - rebuild the CIC-IDS2017 feature matrix exactly as notebook 12 did, so
# the domain classifier sees the same representation the base models saw.
# =============================================================================
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True)
wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
FCOLS=fc.feature_cols(wed)
X=np.column_stack([pd.to_numeric(wed[c], errors='coerce').to_numpy(dtype=float) for c in FCOLS])
X=np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
lab=wed['label'].to_numpy()
print('CIC Wednesday rows:', len(wed), '| features:', len(FCOLS))
print('class counts:', {c:int((lab==c).sum()) for c in ['Benign','DoS']})

REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
missing=[n for n in REAL if not (config.PROC_DIR/f'cic_{n}_srcpool_idx.npy').exists()]
assert not missing, f'missing CIC index files for: {missing}'
print('all five realisation index files present')


CIC Wednesday rows: 477860 | features: 81
class counts: {'Benign': 306105, 'DoS': 171755}
all five realisation index files present


In [4]:
# =============================================================================
# Cell 4 - S_cov,c per class per realisation, then averaged to one row per class
# so the CIC rows are comparable with the other three environments.
# =============================================================================
rows=[]; t0=time.time()
for name in REAL:
    sp=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy')
    tg=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    agg,_,_=scov(X[sp], X[tg], seed=11)
    for cls in ['Benign','DoS']:
        s=sp[lab[sp]==cls]; t=tg[lab[tg]==cls]
        v,ns_,nt_=scov(X[s], X[t], seed=abs(hash((name,cls)))%(2**31))
        nl=scov_null(X[s], X[t], seed=abs(hash((name,cls)))%(2**31))
        rows.append({'dataset':'cicids2017','realization':name,'class':cls,
                     'S_cov_class':v,'S_cov_null':nl,'S_cov_aggregate':agg,
                     'n_src':ns_,'n_tgt':nt_,'estimable':bool(not np.isnan(v))})
        tag=f'{v:.4f}' if not np.isnan(v) else 'not estimable'
        print(f'  {name[:24]:26s} {cls:7s} S_cov,c={tag} (null {nl:.3f})  n={ns_}/{nt_}')
CIC=pd.DataFrame(rows)
print(f'\n{len(CIC)} realisation-class cells | {time.time()-t0:.0f}s')

cic_cls=CIC.groupby(['dataset','class'],as_index=False)[
    ['S_cov_class','S_cov_null','S_cov_aggregate','n_src','n_tgt']].mean()
cic_cls['estimable']=True
cov=pd.read_csv(RD/'coverage_primary_cicids2017.csv')
cov=cov[(np.isclose(cov.alpha,ALPHA))&(cov.protocol=='SHC')]
if 'feasible' in cov.columns: cov=cov[cov['feasible']]
cic_cls=cic_cls.merge(cov.groupby('class',as_index=False)['coverage'].mean(), on='class', how='left')
cic_cls['undercoverage']=(1-ALPHA)-cic_cls['coverage']
# the committed table carries an `excess` column (class shift minus its permutation
# null); supply it here or the CIC rows would be NaN in that column
cic_cls['excess']=cic_cls['S_cov_class']-cic_cls['S_cov_null']
print('\nCIC-IDS2017 averaged over realisations:')
print(cic_cls[['class','S_cov_class','S_cov_null','S_cov_aggregate','coverage','undercoverage']]
      .round(4).to_string(index=False))
print('\n  CIC has two classes, so a rank correlation over two points is meaningless and is')
print('  not reported. The informative check is whether the FAILING class carries the')
print('  higher class-conditional shift.')
fail=cic_cls.loc[cic_cls.undercoverage.idxmax()]
hold=cic_cls.loc[cic_cls.undercoverage.idxmin()]
print(f'    failing: {fail["class"]:7s} coverage {fail.coverage:.4f}  S_cov,c {fail.S_cov_class:.4f}')
print(f'    holding: {hold["class"]:7s} coverage {hold.coverage:.4f}  S_cov,c {hold.S_cov_class:.4f}')
print('    consistent with the other environments:' ,
      'YES' if fail.S_cov_class > hold.S_cov_class else 'NO - this is a counterexample, report it')


  R1_holdout_Slowhttptest    Benign  S_cov,c=0.5002 (null 0.499)  n=22957/153053
  R1_holdout_Slowhttptest    DoS     S_cov,c=0.9994 (null 0.500)  n=25502/1741
  R2_holdout_Slowloris       Benign  S_cov,c=0.5028 (null 0.502)  n=22957/153053
  R2_holdout_Slowloris       DoS     S_cov,c=0.9995 (null 0.501)  n=25163/3998
  R3_holdout_GoldenEye       Benign  S_cov,c=0.4972 (null 0.502)  n=22957/153053
  R3_holdout_GoldenEye       DoS     S_cov,c=1.0000 (null 0.502)  n=24628/7567
  R4_holdout_Slowloris_Slo   Benign  S_cov,c=0.4932 (null 0.501)  n=22957/153053
  R4_holdout_Slowloris_Slo   DoS     S_cov,c=0.9995 (null 0.502)  n=24902/5739
  R5_holdout_GoldenEye_Slo   Benign  S_cov,c=0.5025 (null 0.499)  n=22957/153053
  R5_holdout_GoldenEye_Slo   DoS     S_cov,c=1.0000 (null 0.500)  n=24028/11565

10 realisation-class cells | 201s

CIC-IDS2017 averaged over realisations:
 class  S_cov_class  S_cov_null  S_cov_aggregate  coverage  undercoverage
Benign       0.4992      0.5008           0.7679 

In [5]:
# =============================================================================
# Cell 5 - append and redo the within-dataset comparison across all four.
# =============================================================================
M=pd.read_csv(RD/'class_conditional_scov_vs_coverage.csv')
lost=[c for c in M.columns if c not in cic_cls.columns]
assert not lost, f'CIC rows would be NaN in these existing columns: {lost}'
keep=[c for c in M.columns if c in cic_cls.columns]
M=M[M.dataset!='cicids2017']
M2=pd.concat([M, cic_cls[keep]], ignore_index=True)
assert not M2[keep].isna().any().any(), 'NaNs introduced by the append'
print(f'combined: {len(M2)} class cells across {M2.dataset.nunique()} environments')
print(M2.groupby('dataset').size().to_string())

res=[]
for ds,g in M2.groupby('dataset'):
    spread=float(g.undercoverage.max()-g.undercoverage.min())
    if len(g)>=3:
        r,p=stats.spearmanr(g.S_cov_class, g.undercoverage)
    else:
        r,p=np.nan,np.nan
    res.append({'dataset':ds,'n':len(g),
                'rho':None if np.isnan(r) else round(float(r),3),
                'p':None if np.isnan(p) else round(float(p),4),
                'undercoverage_spread':round(spread,4),
                'aggregate_within':'constant' if g.S_cov_aggregate.nunique()==1 else 'varies'})
R=pd.DataFrame(res)
print('\nWITHIN-DATASET RANK CORRELATION OF S_cov,c WITH UNDERCOVERAGE')
print(R.to_string(index=False))
print('\n  The aggregate is constant within every environment, so it has no within-dataset')
print('  correlation at all. Class-conditional shift is strictly more informative there,')
print('  by construction rather than by contest.')

z=[]
for ds,g in M2.groupby('dataset'):
    if len(g)<3: continue
    z.append(pd.DataFrame({'dataset':ds,'rs':rankdata(g.S_cov_class),
                           'ru':rankdata(g.undercoverage)}))
Z=pd.concat(z, ignore_index=True)
r_all,p_all=stats.spearmanr(Z.rs, Z.ru)
varying=[d for d in R.dataset if float(R[R.dataset==d].undercoverage_spread.iloc[0])>0.05]
Zv=Z[Z.dataset.isin(varying)]
r_var,p_var=(stats.spearmanr(Zv.rs,Zv.ru) if len(Zv)>3 else (np.nan,np.nan))
print(f'\nSTRATIFIED (within-dataset ranks pooled, environments with >=3 classes)')
print(f'  all                       rho={r_all:+.3f} p={p_all:.4f} n={len(Z)}')
if not np.isnan(r_var):
    print(f'  failing environments only rho={r_var:+.3f} p={p_var:.4f} n={len(Zv)}')
print(f'\n  environments with real variation in undercoverage: {varying}')
print('  the all-environment figure is diluted by environments where nothing fails and')
print('  there is therefore no ordering to recover')


combined: 19 class cells across 4 environments
dataset
cicids2017    2
ciciot2023    8
nslkdd        4
ugr16         5

WITHIN-DATASET RANK CORRELATION OF S_cov,c WITH UNDERCOVERAGE
   dataset  n    rho      p  undercoverage_spread aggregate_within
cicids2017  2    NaN    NaN                0.3462         constant
ciciot2023  8 -0.214 0.6103                0.0061         constant
    nslkdd  4  0.800 0.2000                0.8970         constant
     ugr16  5  0.900 0.0374                0.4153         constant

  The aggregate is constant within every environment, so it has no within-dataset
  correlation at all. Class-conditional shift is strictly more informative there,
  by construction rather than by contest.

STRATIFIED (within-dataset ranks pooled, environments with >=3 classes)
  all                       rho=+0.286 p=0.2658 n=17
  failing environments only rho=+0.862 p=0.0028 n=9

  environments with real variation in undercoverage: ['cicids2017', 'nslkdd', 'ugr16']
  the all-

In [6]:
# =============================================================================
# Cell 6 - save and commit. Drive sync can lag behind the write, so the commit
# waits and retries once rather than reporting "nothing to commit" spuriously.
# =============================================================================
M2.to_csv(RD/'class_conditional_scov_vs_coverage.csv', index=False)
CIC.to_csv(RD/'class_conditional_scov_cicids2017_realizations.csv', index=False)
R.to_csv(RD/'class_conditional_scov_within_dataset.csv', index=False)
(RD/'class_conditional_scov_verdict.json').write_text(json.dumps({
 'supersedes':'the pooled comparison in notebook 42, which is withdrawn',
 'why_withdrawn':'aggregate S_cov takes one value per dataset and is constant within a '
                 'dataset, so a pooled rank correlation scores it on between-dataset '
                 'variation alone; the same identification failure as the pooled beta5 model',
 'cicids2017_note':'per-realisation source/target split; S_cov,c computed per realisation '
                   'and averaged, matching how the existing CIC shift measures are built. '
                   'Two classes only, so no within-dataset rank correlation is reported.',
 'within_dataset':R.to_dict('records'),
 'cicids2017_classes':cic_cls.round(5).to_dict('records'),
 'stratified_all':{'rho':float(r_all),'p':float(p_all),'n':int(len(Z))},
 'stratified_failing':{'rho':None if np.isnan(r_var) else float(r_var),
                       'p':None if np.isnan(p_var) else float(p_var),'n':int(len(Zv))},
 'conclusion':'aggregate shift statistics cannot explain class-conditional outcomes; the '
              'class-conditional measure orders them wherever the outcome varies'},
 indent=2, default=str))
print('saved four artefacts')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','step 6c: CIC-IDS2017 added to the class-conditional shift analysis; within-dataset comparison across all four environments')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1:
        print('nothing staged yet, waiting 10s for Drive sync...'); time.sleep(10)
    else:
        print('still nothing to commit; check that the files landed in reports/')
print(git('log','--oneline','-3',show=False).stdout)


saved four artefacts
fatal: Unable to create '/content/drive/MyDrive/CALSHIFT_Research/calshift-research/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
Branch 'main' set up to track remote branch 'main' from 'origin'.
Everything up-to-date
a2e82b4 step 6: class-conditional covariate shift; tests whether S_cov,c orders class-level undercoverage where the aggregate cannot
c0689d4 step 1: immutable results ledger (final_results.json) regenerated from committed reports, plus manuscript number checker
127019b step 1: immutable results ledger (final_results.json) regenerated from committed reports, plus manuscript number checker



In [7]:
# =============================================================================
# Recovery: a previous git process left .git/index.lock behind, so the step 6c
# commit failed and nothing was recorded. The artefacts are written to disk; only
# the commit is missing. Remove the stale lock and commit.
# =============================================================================
import os, subprocess, time
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')
assert PROJECT_ROOT.exists(), 'Drive mount is not healthy; restart the runtime and remount'
os.chdir(PROJECT_ROOT)

lock = PROJECT_ROOT/'.git'/'index.lock'
if lock.exists():
    age = time.time() - lock.stat().st_mtime
    print(f'found .git/index.lock, {age:.0f}s old')
    # only safe to remove if no git process is actually running
    ps = subprocess.run(['pgrep','-a','git'], capture_output=True, text=True).stdout.strip()
    if ps:
        print('a git process IS running; do not remove the lock:'); print(ps)
        raise SystemExit('wait for it to finish, then re-run this cell')
    lock.unlink(); print('stale lock removed')
else:
    print('no lock file present')

def git(*a, show=True):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    if show and (r.stdout or r.stderr): print((r.stdout + r.stderr).strip())
    return r

# confirm the step 6c artefacts really are on disk before committing
expect = ['reports/class_conditional_scov_vs_coverage.csv',
          'reports/class_conditional_scov_cicids2017_realizations.csv',
          'reports/class_conditional_scov_within_dataset.csv',
          'reports/class_conditional_scov_verdict.json']
missing = [f for f in expect if not (PROJECT_ROOT/f).exists()]
print('\nexpected artefacts:', 'all present' if not missing else f'MISSING {missing}')
if missing:
    raise SystemExit('re-run notebook 44 cells 4-6 before committing')

git('add', '-A', show=False)
status = git('status', '--porcelain', show=False).stdout.strip()
if status:
    print('\nstaged changes:'); print(status[:900])
    git('commit', '-m',
        'step 6c: CIC-IDS2017 added to the class-conditional shift analysis; '
        'DoS carries S_cov,c 1.000 while Benign sits at the permutation null, so the '
        'aggregate 0.768 averages total shift with no shift')
    r = git('push', '-u', 'origin', 'main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('\nnothing staged; waiting 10s for Drive sync and retrying once')
    time.sleep(10); git('add', '-A', show=False)
    if git('status', '--porcelain', show=False).stdout.strip():
        git('commit', '-m', 'step 6c: CIC-IDS2017 class-conditional shift')
        git('push')
    else:
        print('still nothing; check that reports/ was written to Drive, not to /content')
print()
git('log', '--oneline', '-3')

found .git/index.lock, 1977s old
stale lock removed

expected artefacts: all present

staged changes:
M  notebooks/42_class_conditional_scov.ipynb
A  notebooks/43_scov_within_dataset.ipynb
A  notebooks/44_cic_class_conditional_scov.ipynb
A  reports/class_conditional_scov_cicids2017_realizations.csv
M  reports/class_conditional_scov_verdict.json
M  reports/class_conditional_scov_vs_coverage.csv
A  reports/class_conditional_scov_within_dataset.csv
[main 3532fec] step 6c: CIC-IDS2017 added to the class-conditional shift analysis; DoS carries S_cov,c 1.000 while Benign sits at the permutation null, so the aggregate 0.768 averages total shift with no shift
 7 files changed, 117 insertions(+), 253 deletions(-)
 create mode 100644 notebooks/43_scov_within_dataset.ipynb
 create mode 100644 notebooks/44_cic_class_conditional_scov.ipynb
 create mode 100644 reports/class_conditional_scov_cicids2017_realizations.csv
 rewrite reports/class_conditional_scov_verdict.json (95%)
 rewrite reports/class_

CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0, stdout='3532fec step 6c: CIC-IDS2017 added to the class-conditional shift analysis; DoS carries S_cov,c 1.000 while Benign sits at the permutation null, so the aggregate 0.768 averages total shift with no shift\na2e82b4 step 6: class-conditional covariate shift; tests whether S_cov,c orders class-level undercoverage where the aggregate cannot\nc0689d4 step 1: immutable results ledger (final_results.json) regenerated from committed reports, plus manuscript number checker\n', stderr='')